# 09 — Blocked OLS and mixed models

Walkthrough of doekit 0.7 analysis: fixed blocks, HC3 standard errors, lack-of-fit, and REML mixed models.

In [1]:
import numpy as np
import pandas as pd
import doekit as ed
print("doekit", ed.__version__)

doekit 0.7.0


## 1. Fixed blocks

Replicate a $2^2$ factorial across two blocks with a large block shift.

In [2]:
base = ed.full_factorial({"A": [-1, 1], "B": [-1, 1]})
mat = pd.concat([base.matrix, base.matrix], ignore_index=True)
design = ed.Design(
    matrix=mat,
    factors=list(base.factors),
    model=ed.Model.main_effects(["A", "B"]),
    metadata={"kind": "FullFactorial"},
)
design = ed.attach_blocks(design, [0, 0, 0, 0, 1, 1, 1, 1], name="block")

rng = np.random.default_rng(0)
y = (
    10
    + 2.0 * design.matrix["A"].to_numpy()
    - 1.5 * design.matrix["B"].to_numpy()
    + 5.0 * (design.matrix["block"].to_numpy() == 1).astype(float)
    + rng.normal(0, 0.05, design.n_runs)
)

fit = ed.fit_linear_model(design, y, blocks="block")
print(fit)
print(ed.anova_table(fit))

FitResult(R^2=1.0000, dof=4, cov_type='nonrobust', blocks='block')
          term  estimate  std_error  t_value    p_value
0  (Intercept)    10.009   0.012405   806.84 1.4158e-11
1            A    1.9984   0.008772   227.82 2.2271e-09
2            B   -1.4801   0.008772  -168.74 7.3998e-09
3     block[1]    5.0167   0.017544   285.95 8.9731e-10
       term  df             F       p_value
0         A   1  51901.494878  2.227080e-09
1         B   1  28471.820689  7.399785e-09
2  block[1]   1  81768.801845  8.973060e-10
3  Residual   4           NaN           NaN


## 2. Robust SE (HC3)

In [3]:
pb = ed.plackett_burman(5)
scale = 0.05 + 0.4 * (pb.matrix["factor1"].to_numpy() > 0)
y_het = 1.0 + 2.0 * pb.matrix["factor1"].to_numpy() + rng.normal(0, 1, pb.n_runs) * scale
print(ed.fit_linear_model(pb, y_het, cov_type="nonrobust").summary_frame())
print(ed.fit_linear_model(pb, y_het, cov_type="HC3").summary_frame())

      term  estimate  std_error   t_value   p_value
0  factor1  1.738085   0.710896  2.444923  0.247168
1  factor2 -0.070184   0.710896 -0.098727  0.937352
2  factor3 -0.060268   0.710896 -0.084778  0.946157
3  factor4  0.124515   0.710896  0.175152  0.889615
4  factor5  0.127928   0.710896  0.179953  0.886652
5   dummy1  0.044798   0.710896  0.063017  0.959935
6   dummy2  0.067551   0.710896  0.095022  0.939688
      term  estimate  std_error   t_value   p_value
0  factor1  1.738085   2.010716  0.864411  0.387362
1  factor2 -0.070184   2.010716 -0.034905  0.972155
2  factor3 -0.060268   2.010716 -0.029974  0.976088
3  factor4  0.124515   2.010716  0.061926  0.950622
4  factor5  0.127928   2.010716  0.063623  0.949270
5   dummy1  0.044798   2.010716  0.022280  0.982225
6   dummy2  0.067551   2.010716  0.033595  0.973200


## 3. Lack of fit (center replicates)

In [4]:
fac = ed.full_factorial({"A": [-1, 1], "B": [-1, 1]})
centers = pd.DataFrame({"A": [0, 0, 0], "B": [0, 0, 0]})
mat2 = pd.concat([fac.matrix, centers], ignore_index=True)
# En +/-1, A^2 y B^2 son colineales (misma columna). Para LOF usamos un modelo
# subespecificado (solo efectos principales) con centros replicados = error puro;
# la curvatura en y aparece como falta de ajuste.
d2 = ed.Design(
    matrix=mat2,
    factors=list(fac.factors),
    model=ed.Model.main_effects(["A", "B"]),
)
y2 = (
    5
    + 1.5 * d2.matrix["A"]
    - 0.8 * d2.matrix["B"]
    + 0.5 * d2.matrix["A"] ** 2
    + rng.normal(0, 0.1, d2.n_runs)
)
print(ed.lack_of_fit(d2, y2))

        source  df        ss        ms          F   p_value
0  lack_of_fit   2  0.421720  0.210860  19.023078  0.049942
1   pure_error   2  0.022169  0.011084        NaN       NaN
2     residual   4  0.443889  0.110972        NaN       NaN


## 4. Mixed model (random batch intercept)

In [5]:
n_g, n_per = 4, 6
n = n_g * n_per
group = np.repeat(np.arange(n_g), n_per)
x = rng.uniform(-1, 1, n)
re = rng.normal(0, 1.5, n_g)
y3 = 2.0 + 1.2 * x + re[group] + rng.normal(0, 0.3, n)
dm = ed.Design(
    matrix=pd.DataFrame({"x": x, "batch": group}),
    factors=[ed.ContinuousFactor("x", -1, 1)],
    model=ed.Model.parse("0 ~ x"),
)
mix = ed.fit_mixed_model(dm, y3, groups="batch")
print(mix)
print("re_var", mix.re_var)
print(mix.to_dict()["schema"])

MixedFitResult(method='reml', groups='batch', n_groups=4, sigma2=0.11065)
          term  estimate  std_error  z_value    p_value
0  (Intercept)    4.0731    0.52862   7.7053 1.3057e-14
1            x    1.1131    0.15356   7.2487 4.2078e-13
re_var {'Intercept': 1.0973368188418309}
doekit.MixedFitResult/1


## Puente: split-plot (generación + mixed)

`split_plot_design` escribe `whole_plot_id` en la matriz; el análisis correcto es
mixed LM con ese grouping (no OLS i.i.d.).


In [6]:
spd = ed.split_plot_design(
    whole_plot=[ed.ContinuousFactor("temp", -1, 1)],
    subplot=[ed.ContinuousFactor("pH", -1, 1),
             ed.ContinuousFactor("agit", -1, 1)],
    whole_plot_reps=2, seed=0,
)
print(spd.matrix.head())
print("metadata hard_to_change:", spd.metadata.get("hard_to_change"))
rng = np.random.default_rng(0)
# WP effect + SP noise (ilustrativo)
wp = spd.matrix.groupby("whole_plot_id").ngroup().to_numpy()
y_spd = 0.8 * spd.matrix["temp"].to_numpy() + 0.4 * wp + rng.normal(0, 0.3, spd.n_runs)
mix = ed.fit_mixed_model(spd, y_spd, groups="whole_plot_id")
print(mix.summary_frame().head())


   temp  pH  agit  whole_plot_id
0    -1  -1    -1              0
1    -1   1    -1              0
2    -1  -1     1              0
3    -1   1     1              0
4     1  -1    -1              1
metadata hard_to_change: ['temp']
          term  estimate  std_error   z_value       p_value
0  (Intercept)  0.520178   0.157945  3.293419  9.897673e-04
1         temp  0.988134   0.157945  6.256201  3.944686e-10
2           pH  0.046308   0.044487  1.040952  2.978980e-01
3         agit  0.096182   0.044487  2.162041  3.061505e-02
